In [17]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.tree import DecisionTreeClassifier, export_text

# Đăng ký thiết bị chạy
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load dữ liệu chuỗi khám bệnh nhân
DATA_PROCESSED_PATH = "../data/processed/patient_sequences.json"
with open(DATA_PROCESSED_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

sequences = data["sequences"]
splits = data["splits"]

# Load mô hình Hộp đen Doctor AI đã huấn luyện từ Giai đoạn 2
MODEL_PATH = "../models/doctor_ai_best.pt"
checkpoint = torch.load(MODEL_PATH, map_location=device)

code2idx = checkpoint["code2idx"]
idx2code = checkpoint["idx2code"]
vocab_size = checkpoint["vocab_size"]

# Re-instantiate Doctor AI Architecture
class DoctorAI(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128):
        super(DoctorAI, self).__init__()
        self.embedding = nn.Linear(vocab_size, embed_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x):
        emb = self.dropout(self.relu(self.embedding(x)))
        gru_out, _ = self.gru(emb)
        return self.fc_out(gru_out)

black_box_model = DoctorAI(vocab_size=vocab_size).to(device)
black_box_model.load_state_dict(checkpoint["model_state_dict"])
black_box_model.eval()

print("Đã load thành công Mô hình Hộp đen Doctor AI và Dữ liệu chuỗi!")

Đã load thành công Mô hình Hộp đen Doctor AI và Dữ liệu chuỗi!


In [18]:
def get_icd_prefix(code_str):
    """Lấy tiền tố nhóm cha (Least Common Superconcept) của mã ICD-9."""
    if "." in code_str:
        return code_str.split(".")
    return code_str[:3] if len(code_str) >= 3 else code_str

def wualmer_similarity(code1, code2):
    """
    Tính độ tương đồng Wu-Palmer (WuP) đơn giản hóa giữa 2 mã ICD-9.
    WuP = 1.0 nếu cùng mã hoàn toàn.
    WuP = 0.75 nếu cùng nhóm tiền tố 3 chữ số (Superconcept).
    WuP = 0.0 nếu khác nhóm hoàn toàn.
    """
    if code1 == code2:
        return 1.0
    prefix1 = get_icd_prefix(code1)
    prefix2 = get_icd_prefix(code2)
    if prefix1 == prefix2:
        return 0.75
    return 0.0

print("Thử nghiệm Wu-Palmer Similarity:")
print(" • (276.1 vs 276.1):", wualmer_similarity("276.1", "276.1"))
print(" • (276.1 vs 276.2 - Cùng nhóm 276):", wualmer_similarity("276.1", "276.2"))
print(" • (276.1 vs 530.1 - Khác nhóm):", wualmer_similarity("276.1", "530.1"))

Thử nghiệm Wu-Palmer Similarity:
 • (276.1 vs 276.1): 1.0
 • (276.1 vs 276.2 - Cùng nhóm 276): 0.0
 • (276.1 vs 530.1 - Khác nhóm): 0.0


In [19]:
def visit_distance_levenshtein(visit1, visit2):
    """Tính khoảng cách Levenshtein giữa 2 lượt khám dựa trên chi phí edit (1 - WuP)."""
    if not visit1 and not visit2:
        return 0.0
    if not visit1:
        return float(len(visit2))
    if not visit2:
        return float(len(visit1))
    
    n, m = len(visit1), len(visit2)
    # Ma trận khoảng cách kích thước (n+1) x (m+1)
    dp = np.zeros((n + 1, m + 1), dtype=np.float32)
    
    # Khởi tạo giá trị đúng theo chuẩn ma trận 2D NumPy
    for i in range(n + 1):
        dp[i, 0] = float(i)
    for j in range(m + 1):
        dp[0, j] = float(j)
        
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 1.0 - wualmer_similarity(visit1[i-1], visit2[j-1])
            dp[i, j] = min(
                dp[i-1, j] + 1.0,      # Deletion
                dp[i, j-1] + 1.0,      # Insertion
                dp[i-1, j-1] + cost    # Substitution
            )
    return dp[n, m]

def dtw_patient_distance(seq1, seq2):
    """Tính khoảng cách Dynamic Time Warping (DTW) giữa 2 chuỗi bệnh nhân."""
    N, M = len(seq1), len(seq2)
    dtw_matrix = np.zeros((N + 1, M + 1), dtype=np.float32)
    dtw_matrix[0, 1:] = float("inf")
    dtw_matrix[1:, 0] = float("inf")
    
    for i in range(1, N + 1):
        for j in range(1, M + 1):
            cost = visit_distance_levenshtein(seq1[i-1], seq2[j-1])
            dtw_matrix[i, j] = cost + min(
                dtw_matrix[i-1, j],    # Insertion
                dtw_matrix[i, j-1],    # Deletion
                dtw_matrix[i-1, j-1]   # Match/Mismatch
            )
    return dtw_matrix[N, M]

def find_k_real_neighbors(target_patient_id, sequences, train_ids, k=10):
    """Tìm k láng giềng thực sự gần nhất trong tập Train."""
    target_seq = sequences[target_patient_id]
    distances = []
    
    for pid in train_ids:
        if pid == target_patient_id:
            continue
        dist = dtw_patient_distance(target_seq, sequences[pid])
        distances.append((pid, dist))
        
    distances.sort(key=lambda x: x[1])
    return [pid for pid, _ in distances[:k]]
    """Tính khoảng cách Dynamic Time Warping (DTW) giữa 2 chuỗi bệnh nhân."""
    N, M = len(seq1), len(seq2)
    dtw_matrix = np.zeros((N + 1, M + 1))
    dtw_matrix[0, 1:] = float("inf")
    dtw_matrix[1:, 0] = float("inf")
    
    for i in range(1, N + 1):
        for j in range(1, M + 1):
            cost = visit_distance_levenshtein(seq1[i-1], seq2[j-1])
            dtw_matrix[i, j] = cost + min(
                dtw_matrix[i-1, j],    # Insertion
                dtw_matrix[i, j-1],    # Deletion
                dtw_matrix[i-1, j-1]   # Match/Mismatch
            )
    return dtw_matrix[N, M]

def find_k_real_neighbors(target_patient_id, sequences, train_ids, k=10):
    """Tìm k láng giềng thực sự gần nhất trong tập Train."""
    target_seq = sequences[target_patient_id]
    distances = []
    
    for pid in train_ids:
        if pid == target_patient_id:
            continue
        dist = dtw_patient_distance(target_seq, sequences[pid])
        distances.append((pid, dist))
        
    # Sắp xếp theo khoảng cách dist (chỉ số x[1])
    distances.sort(key=lambda x: x[1])
    return [pid for pid, _ in distances[:k]]

In [20]:
def ontological_perturbation(patient_sequence, num_samples=50):
    """
    Tạo các mẫu bệnh nhân tổng hợp (synthetic neighbors) bằng cách làm mờ (masking)
    tất cả các mã ICD-9 có cùng nhóm tiền tố cha.
    """
    synthetic_samples = [patient_sequence] # Giữ bản gốc
    
    # Lấy toàn bộ các tiền tố ICD-9 xuất hiện trong lịch sử bệnh nhân
    all_prefixes = set()
    for visit in patient_sequence:
        for code in visit:
            all_prefixes.add(get_icd_prefix(code))
            
    all_prefixes = list(all_prefixes)
    if not all_prefixes:
        return synthetic_samples
        
    for _ in range(num_samples):
        # Chọn ngẫu nhiên 1 hoặc nhiều tiền tố để mask
        masked_prefix = np.random.choice(all_prefixes)
        
        new_seq = []
        for visit in patient_sequence:
            new_visit = [code for code in visit if get_icd_prefix(code) != masked_prefix]
            new_seq.append(new_visit)
        synthetic_samples.append(new_seq)
        
    return synthetic_samples

def temporal_encoding(patient_sequence, code2idx, vocab_size):
    """
    Mã hóa chuỗi khám thành vector phẳng thưa:
    EN(c, P) = sum( (1/2)^(n - i + 1) ) cho các lượt khám mà c xuất hiện.
    """
    n_visits = len(patient_sequence)
    flat_vector = np.zeros(vocab_size, dtype=np.float32)
    
    for i, visit in enumerate(patient_sequence):
        weight = (0.5) ** (n_visits - i) # 0.5 cho lượt cuối, 0.25 cho lượt áp cuối,...
        for code in visit:
            if code in code2idx:
                flat_vector[code2idx[code]] += weight
                
    return flat_vector

In [21]:
def explain_patient_doctor_xai(target_pid, sequences, train_ids, black_box_model, code2idx, idx2code, vocab_size, k_neighbors=10, num_synth=100):
    """
    Quy trình giải thích Doctor XAI đầy đủ cho 1 bệnh nhân.
    """
    target_seq = sequences[target_pid]
    
    # 1. Tìm k láng giềng thực sự
    real_neighbor_ids = find_k_real_neighbors(target_pid, sequences, train_ids, k=k_neighbors)
    
    # 2. Tạo tập láng giềng tổng hợp (Synthetic Neighborhood)
    all_synth_seqs = []
    for pid in [target_pid] + real_neighbor_ids:
        synth_seqs = ontological_perturbation(sequences[pid], num_samples=num_synth // len(real_neighbor_ids))
        all_synth_seqs.extend(synth_seqs)
        
    # 3. Mã hóa thời gian (Temporal Encoding) & Dự đoán nhãn từ Black-box
    X_flat = []
    Y_black_box = []
    
    with torch.no_grad():
        for seq in all_synth_seqs:
            if not seq or len(seq) == 0:
                continue
            flat_v = temporal_encoding(seq, code2idx, vocab_size)
            X_flat.append(flat_v)
            
            # Biến đổi thành Tensor để truyền qua mô hình Hộp đen
            seq_matrix = torch.zeros((1, len(seq), vocab_size), dtype=torch.float32).to(device)
            for t, visit in enumerate(seq):
                for code in visit:
                    if code in code2idx:
                        seq_matrix[0, t, code2idx[code]] = 1.0
                        
            logits = black_box_model(seq_matrix)
            probs = torch.sigmoid(logits[0, -1, :]).cpu().numpy() # Lấy xác suất ở lượt khám cuối cùng
            pred_binary = (probs >= 0.3).astype(int) # Ngưỡng dự đoán nhãn
            Y_black_box.append(pred_binary)
            
    X_flat = np.array(X_flat)
    Y_black_box = np.array(Y_black_box)
    
    # 4. Huấn luyện Cây quyết định Cục bộ (Local Decision Tree)
    explainer_dt = DecisionTreeClassifier(max_depth=4, random_state=42)
    explainer_dt.fit(X_flat, Y_black_box)
    
    # 5. Trích xuất dự đoán của Hộp đen cho chính bệnh nhân gốc
    target_flat = temporal_encoding(target_seq, code2idx, vocab_size).reshape(1, -1)
    target_pred_bb = explainer_dt.predict(target_flat)
    
    return explainer_dt, target_seq, target_pred_bb

In [22]:
# Chọn 1 bệnh nhân từ tập Test để chạy thử nghiệm giải thích
sample_test_pid = splits["test"][0]

print(f"Bắt đầu trích xuất giải thích cho Bệnh nhân Test ID: {sample_test_pid}")
print(f"Lịch sử khám bệnh gốc (ICD-9): {sequences[sample_test_pid]}")

dt_model, target_seq, target_pred = explain_patient_doctor_xai(
    target_pid=sample_test_pid,
    sequences=sequences,
    train_ids=splits["train"][:500], # Chọn 500 bệnh nhân train đầu để chạy nhanh
    black_box_model=black_box_model,
    code2idx=code2idx,
    idx2code=idx2code,
    vocab_size=vocab_size,
    k_neighbors=5,
    num_synth=50
)

# In ra cây luật quyết định cục bộ
feature_names = [idx2code.get(i, f"Code_{i}") for i in range(vocab_size)]
tree_rules = export_text(dt_model, feature_names=feature_names)

print("\n TẬP LUẬT GIẢI THÍCH TRÍCH XUẤT TỪ CÂY QUYẾT ĐỊNH (DOCTOR XAI RULE):")
print(tree_rules)

Bắt đầu trích xuất giải thích cho Bệnh nhân Test ID: 10012853
Lịch sử khám bệnh gốc (ICD-9): [['4150', '5990', '5119', '4280', '0413', '27669', '5853', '25000', '42731', '7842', '2411', '2724', '7804', '40390', 'V5861', '4439'], ['5849', '42731', '25000', '59654', '2411', '27651', '2809', '2724', '27800', '5589', '40390', '5853', '7919', '4439', '4414', '73300', '3051', 'V8521', 'V5866', 'V1255', 'V5861', 'V1254']]



 TẬP LUẬT GIẢI THÍCH TRÍCH XUẤT TỪ CÂY QUYẾT ĐỊNH (DOCTOR XAI RULE):
|--- 42731 <= 0.38
|   |--- 4019 <= 0.50
|   |   |--- class: 0
|   |--- 4019 >  0.50
|   |   |--- 3484 <= 0.12
|   |   |   |--- 78062 <= 0.12
|   |   |   |   |--- class: 0
|   |   |   |--- 78062 >  0.12
|   |   |   |   |--- class: 0
|   |   |--- 3484 >  0.12
|   |   |   |--- class: 0
|--- 42731 >  0.38
|   |--- 2724 <= 0.38
|   |   |--- V1582 <= 0.38
|   |   |   |--- 4019 <= 0.38
|   |   |   |   |--- class: 0
|   |   |   |--- 4019 >  0.38
|   |   |   |   |--- class: 0
|   |   |--- V1582 >  0.38
|   |   |   |--- class: 0
|   |--- 2724 >  0.38
|   |   |--- class: 0

